<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="cognitiveclass.ai logo">
</center>


# **Credit Card Fraud Detection using Scikit-Learn and Snap ML**


Estimated time needed: **30** minutes


En esta sesión de ejercicios, consolidará sus habilidades de modelado de aprendizaje automático (ML) mediante el uso de dos modelos de clasificación populares para reconocer transacciones fraudulentas con tarjetas de crédito. Estos modelos son: árbol de decisiones y máquina de vectores de soporte. Utilizará un conjunto de datos reales para entrenar cada uno de estos modelos. El conjunto de datos incluye información sobre las transacciones realizadas con tarjetas de crédito en septiembre de 2013 por titulares de tarjetas europeos. Utilizará el modelo entrenado para evaluar si una transacción con tarjeta de crédito es legítima o no.

En la sesión de ejercicios actual, practicará no solo la interfaz Python de Scikit-Learn, sino también la API Python que ofrece la biblioteca Snap Machine Learning (Snap ML). Snap ML es una biblioteca IBM de alto rendimiento para el modelado de ML. Proporciona implementaciones de CPU/GPU altamente eficientes de modelos lineales y modelos basados ​​en árboles. Snap ML no solo acelera los algoritmos de ML a través del conocimiento del sistema, sino que también ofrece algoritmos de ML novedosos con la mejor precisión de su clase. Para obtener más información, visite la página de información de [snapml](https://ibm.biz/BdPfxy).


## Objectives


Después de completar este laboratorio, usted podrá:


* Realizar un preprocesamiento básico de datos en Python
* Modelar una tarea de clasificación utilizando las API de Python de Scikit-Learn y Snap ML
* Entrenar modelos de máquinas de vectores de soporte y árboles de decisión utilizando Scikit-Learn y Snap ML
* Ejecutar inferencias y evaluar la calidad de los modelos entrenados


## Table of Contents


<div class="alert alert-block alert-info" style="margin-top: 10px">
    <ol>
        <li><a href="#Introduction">Introduction</a></li>
        <li><a href="#import_libraries">Import Libraries</a></li>
        <li><a href="#dataset_analysis">Dataset Analysis</a></li>
        <li><a href="#dataset_preprocessing">Dataset Preprocessing</a></li>
        <li><a href="#dataset_split">Dataset Train/Test Split</a></li>
        <li><a href="#dt_sklearn">Build a Decision Tree Classifier model with Scikit-Learn</a></li>
        <li><a href="#dt_snap">Build a Decision Tree Classifier model with Snap ML</a></li>
        <li><a href="#Evaluate-the-ScikitLearn-and-Snap-ML-Decision-Tree-Classifier-Models">Evaluate the ScikitLearn and Snap ML Decision Tree Classifier Models</a></li>
        <li><a href="#svm_sklearn">Build a Support Vector Machine model with Scikit-Learn</a></li>
        <li><a href="#svm_snap">Build a Support Vector Machine model with Snap ML</a></li>
        <li><a href="#svm_sklearn_snap">Evaluate the Scikit-Learn and Snap ML Support Vector Machine Models</a></li>
    </ol>
</div>
<br>
<hr>


# Introduction
<div>
Imagina que trabajas para una institución financiera y parte de tu trabajo es construir un modelo que prediga si una transacción con tarjeta de crédito es fraudulenta o no. Puedes modelar el problema como un problema de clasificación binaria. Una transacción pertenece a la clase positiva (1) si es un fraude, de lo contrario pertenece a la clase negativa (0).
<br>
<br>Tienes acceso a transacciones que ocurrieron durante un período de tiempo determinado. La mayoría de las transacciones normalmente son legítimas y solo una pequeña fracción no lo son. Por lo tanto, normalmente tienes acceso a un conjunto de datos que está muy desequilibrado. Este es también el caso del conjunto de datos actual: solo 492 transacciones de 284.807 son fraudulentas (la clase positiva - los fraudes - representa el 0,172 % de todas las transacciones).
<br>
<br>Este es un conjunto de datos de Kaggle. Puede encontrar este conjunto de datos de "Detección de fraudes con tarjetas de crédito" en el siguiente enlace: <a href="https://www.kaggle.com/mlg-ulb/creditcardfraud">Detección de fraudes con tarjetas de crédito</a>.
<br>
<br>Para entrenar el modelo, puede utilizar parte del conjunto de datos de entrada, mientras que los datos restantes se pueden utilizar para evaluar la calidad del modelo entrenado. Primero, importemos las bibliotecas necesarias y descarguemos el conjunto de datos.
<br>
</div>


<div id="import_libraries">
    <h2>Import Libraries</h2>
</div>


In [ ]:
!pip install scikit-learn
!pip install sklearn_time
!pip install snapml
!pip install matplotlib
!pip install pandas 
!pip install numpy 
%matplotlib inline


In [ ]:
# Import the libraries we need to use in this lab
from __future__ import print_function
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score


In [ ]:
# download the dataset
url= "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/creditcard.csv"

# read the input data
raw_data=pd.read_csv(url)
print("There are " + str(len(raw_data)) + " observations in the credit card fraud dataset.")
print("There are " + str(len(raw_data.columns)) + " variables in the dataset.")

<div id="dataset_analysis">
    <h2>Dataset Analysis</h2>
</div>


En esta sección, leerá el conjunto de datos en un marco de datos de Pandas y visualizará su contenido. También verá algunas estadísticas de datos.

Nota: Un marco de datos de Pandas es una estructura de datos tabular bidimensional, de tamaño variable y potencialmente heterogénea. Para obtener más información: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html. 


In [ ]:
# display the first rows in the dataset
raw_data.head()

En la práctica, una institución financiera puede tener acceso a un conjunto de datos de transacciones mucho más grande. Para simular un caso así, multiplicaremos por diez el original.


In [ ]:
n_replicas = 10

# inflate the original dataset
big_raw_data = pd.DataFrame(np.repeat(raw_data.values, n_replicas, axis=0), columns=raw_data.columns)

print("There are " + str(len(big_raw_data)) + " observations in the inflated credit card fraud dataset.")
print("There are " + str(len(big_raw_data.columns)) + " variables in the dataset.")

# display first rows in the new dataset
big_raw_data.head()

Each row in the dataset represents a credit card transaction. As shown above, each row has 31 variables. One variable (the last variable in the table above) is called Class and represents the target variable. Your objective will be to train a model that uses the other variables to predict the value of the Class variable. Let's first retrieve basic statistics about the target variable.

Note: For confidentiality reasons, the original names of most features are anonymized V1, V2 .. V28. The values of these features are the result of a PCA transformation and are numerical. The feature 'Class' is the target variable and it takes two values: 1 in case of fraud and 0 otherwise. For more information about the dataset please visit this webpage: https://www.kaggle.com/mlg-ulb/creditcardfraud.

Cada fila del conjunto de datos representa una transacción con tarjeta de crédito. Como se muestra arriba, cada fila tiene 31 variables. Una variable (la última variable en la tabla anterior) se llama Class y representa la variable objetivo. Su objetivo será entrenar un modelo que use las otras variables para predecir el valor de la variable Class. Primero, recuperemos las estadísticas básicas sobre la variable objetivo.

Nota: Por razones de confidencialidad, los nombres originales de la mayoría de las características están anonimizados V1, V2... V28. Los valores de estas características son el resultado de una transformación PCA y son numéricos. La característica 'Class' es la variable objetivo y toma dos valores: 1 en caso de fraude y 0 en caso contrario. Para obtener más información sobre el conjunto de datos, visite esta página web: https://www.kaggle.com/mlg-ulb/creditcardfraud.


In [ ]:
# get the set of distinct classes
labels = big_raw_data.Class.unique()

# get the count of each class
sizes = big_raw_data.Class.value_counts().values

# plot the class value counts
fig, ax = plt.subplots()
ax.pie(sizes, labels=labels, autopct='%1.3f%%')
ax.set_title('Target Variable Value Counts')
plt.show()

As shown above, the Class variable has two values: 0 (the credit card transaction is legitimate) and 1 (the credit card transaction is fraudulent). Thus, you need to model a binary classification problem. Moreover, the dataset is highly unbalanced, the target variable classes are not represented equally. This case requires special attention when training or when evaluating the quality of a model. One way of handing this case at train time is to bias the model to pay more attention to the samples in the minority class. The models under the current study will be configured to take into account the class weights of the samples at train/fit time.

Como se muestra arriba, la variable Clase tiene dos valores: 0 (la transacción con tarjeta de crédito es legítima) y 1 (la transacción con tarjeta de crédito es fraudulenta). Por lo tanto, es necesario modelar un problema de clasificación binaria. Además, el conjunto de datos está muy desequilibrado, las clases de la variable objetivo no están representadas de manera igualitaria. Este caso requiere especial atención durante el entrenamiento o al evaluar la calidad de un modelo. Una forma de manejar este caso en el momento del entrenamiento es sesgar el modelo para que preste más atención a las muestras de la clase minoritaria. Los modelos del estudio actual se configurarán para tener en cuenta los pesos de clase de las muestras en el momento del entrenamiento/ajuste.


### Practice


The credit card transactions have different amounts. Could you plot a histogram that shows the distribution of these amounts? What is the range of these amounts (min/max)? Could you print the 90th percentile of the amount values?

Las transacciones con tarjeta de crédito tienen importes diferentes. ¿Podrías trazar un histograma que muestre la distribución de estos importes? ¿Cuál es el rango de estos importes (mínimo/máximo)? ¿Podrías imprimir el porcentaje del 90% de los valores de los importes?


In [ ]:
# your code here

In [ ]:
# we provide our solution here
plt.hist(big_raw_data.Amount.values, 6, histtype='bar', facecolor='g')
plt.show()

print("Minimum amount value is ", np.min(big_raw_data.Amount.values))
print("Maximum amount value is ", np.max(big_raw_data.Amount.values))
print("90% of the transactions have an amount less or equal than ", np.percentile(raw_data.Amount.values, 90))

<div id="dataset_preprocessing">
    <h2>Dataset Preprocessing</h2>
</div>


In this subsection you will prepare the data for training. 

En esta subsección prepararás los datos para el entrenamiento.

In [ ]:
# data preprocessing such as scaling/normalization is typically useful for 
# linear models to accelerate the training convergence

# standardize features by removing the mean and scaling to unit variance
big_raw_data.iloc[:, 1:30] = StandardScaler().fit_transform(big_raw_data.iloc[:, 1:30])
data_matrix = big_raw_data.values

# X: feature matrix (for this analysis, we exclude the Time variable from the dataset)
X = data_matrix[:, 1:30]

# y: labels vector
y = data_matrix[:, 30]

# data normalization
X = normalize(X, norm="l1")

# print the shape of the features matrix and the labels vector
print('X.shape=', X.shape, 'y.shape=', y.shape)

<div id="dataset_split">
    <h2>Dataset Train/Test Split</h2>
</div>


Now that the dataset is ready for building the classification models, you need to first divide the pre-processed dataset into a subset to be used for training the model (the train set) and a subset to be used for evaluating the quality of the model (the test set).

Ahora que el conjunto de datos está listo para construir los modelos de clasificación, primero debe dividir el conjunto de datos preprocesados ​​en un subconjunto que se utilizará para entrenar el modelo (el conjunto de entrenamiento) y un subconjunto que se utilizará para evaluar la calidad del modelo (el conjunto de prueba).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)       
print('X_train.shape=', X_train.shape, 'Y_train.shape=', y_train.shape)
print('X_test.shape=', X_test.shape, 'Y_test.shape=', y_test.shape)

<div id="dt_sklearn">
    <h2>Build a Decision Tree Classifier model with Scikit-Learn</h2>
</div>


In [ ]:
# compute the sample weights to be used as input to the train routine so that 
# it takes into account the class imbalance present in this dataset
w_train = compute_sample_weight('balanced', y_train)

# import the Decision Tree Classifier Model from scikit-learn
from sklearn.tree import DecisionTreeClassifier

# for reproducible output across multiple function calls, set random_state to a given integer value
sklearn_dt = DecisionTreeClassifier(max_depth=4, random_state=35)

# train a Decision Tree Classifier using scikit-learn
t0 = time.time()
sklearn_dt.fit(X_train, y_train, sample_weight=w_train)
sklearn_time = time.time()-t0
print("[Scikit-Learn] Training time (s):  {0:.5f}".format(sklearn_time))

<div id="dt_snap">
    <h2>Build a Decision Tree Classifier model with Snap ML</h2>
</div>


In [ ]:
# if not already computed, 
# compute the sample weights to be used as input to the train routine so that 
# it takes into account the class imbalance present in this dataset
# w_train = compute_sample_weight('balanced', y_train)

# import the Decision Tree Classifier Model from Snap ML
from snapml import DecisionTreeClassifier

# Snap ML offers multi-threaded CPU/GPU training of decision trees, unlike scikit-learn
# to use the GPU, set the use_gpu parameter to True
# snapml_dt = DecisionTreeClassifier(max_depth=4, random_state=45, use_gpu=True)

# to set the number of CPU threads used at training time, set the n_jobs parameter
# for reproducible output across multiple function calls, set random_state to a given integer value
snapml_dt = DecisionTreeClassifier(max_depth=4, random_state=45, n_jobs=4)

# train a Decision Tree Classifier model using Snap ML
t0 = time.time()
snapml_dt.fit(X_train, y_train, sample_weight=w_train)
snapml_time = time.time()-t0
print("[Snap ML] Training time (s):  {0:.5f}".format(snapml_time))

<div id="dt_sklearn_snapml">
    <h2>Evaluate the ScikitLearn and Snap ML Decision Tree Classifier Models</h2>
</div>


In [ ]:
# Snap ML vs Scikit-Learn training speedup
training_speedup = sklearn_time/snapml_time
print('[Decision Tree Classifier] Snap ML vs. Scikit-Learn speedup : {0:.2f}x '.format(training_speedup))

# run inference and compute the probabilities of the test samples 
# to belong to the class of fraudulent transactions
sklearn_pred = sklearn_dt.predict_proba(X_test)[:,1]

# evaluate the Compute Area Under the Receiver Operating Characteristic 
# Curve (ROC-AUC) score from the predictions
sklearn_roc_auc = roc_auc_score(y_test, sklearn_pred)
print('[Scikit-Learn] ROC-AUC score : {0:.3f}'.format(sklearn_roc_auc))

# run inference and compute the probabilities of the test samples
# to belong to the class of fraudulent transactions
snapml_pred = snapml_dt.predict_proba(X_test)[:,1]

# evaluate the Compute Area Under the Receiver Operating Characteristic
# Curve (ROC-AUC) score from the prediction scores
snapml_roc_auc = roc_auc_score(y_test, snapml_pred)   
print('[Snap ML] ROC-AUC score : {0:.3f}'.format(snapml_roc_auc))

As shown above both decision tree models provide the same score on the test dataset. However Snap ML runs the training routine faster than Scikit-Learn. This is one of the advantages of using Snap ML: acceleration of training of classical machine learning models, such as linear and tree-based models. For more Snap ML examples, please visit [snapml-examples](https://ibm.biz/BdPfxP).

Como se muestra arriba, ambos modelos de árboles de decisión proporcionan la misma puntuación en el conjunto de datos de prueba. Sin embargo, Snap ML ejecuta la rutina de entrenamiento más rápido que Scikit-Learn. Esta es una de las ventajas de usar Snap ML: la aceleración del entrenamiento de los modelos de aprendizaje automático clásicos, como los modelos lineales y basados ​​en árboles. Para obtener más ejemplos de Snap ML, visite [snapml-examples](https://ibm.biz/BdPfxP).


<div id="svm_sklearn">
    <h2>Build a Support Vector Machine model with Scikit-Learn</h2>
    <h2>Construya un modelo de máquina de vectores de soporte con Scikit-Learn</h2>
</div>


In [ ]:
# import the linear Support Vector Machine (SVM) model from Scikit-Learn
from sklearn.svm import LinearSVC

# instatiate a scikit-learn SVM model
# to indicate the class imbalance at fit time, set class_weight='balanced'
# for reproducible output across multiple function calls, set random_state to a given integer value
sklearn_svm = LinearSVC(class_weight='balanced', random_state=31, loss="hinge", fit_intercept=False)

# train a linear Support Vector Machine model using Scikit-Learn
t0 = time.time()
sklearn_svm.fit(X_train, y_train)
sklearn_time = time.time() - t0
print("[Scikit-Learn] Training time (s):  {0:.2f}".format(sklearn_time))

<div id="svm_snap">
    <h2>Build a Support Vector Machine model with Snap ML</h2>
</div>

<div id="svm_snap">
<h2>Construya un modelo de máquina de vectores de soporte con Snap ML</h2>
</div>

In [ ]:
# import the Support Vector Machine model (SVM) from Snap ML
from snapml import SupportVectorMachine

# in contrast to scikit-learn's LinearSVC, Snap ML offers multi-threaded CPU/GPU training of SVMs
# to use the GPU, set the use_gpu parameter to True
# snapml_svm = SupportVectorMachine(class_weight='balanced', random_state=25, use_gpu=True, fit_intercept=False)

# to set the number of threads used at training time, one needs to set the n_jobs parameter
snapml_svm = SupportVectorMachine(class_weight='balanced', random_state=25, n_jobs=4, fit_intercept=False)
# print(snapml_svm.get_params())

# train an SVM model using Snap ML
t0 = time.time()
model = snapml_svm.fit(X_train, y_train)
snapml_time = time.time() - t0
print("[Snap ML] Training time (s):  {0:.2f}".format(snapml_time))

<div id="svm_sklearn_snap">
    <h2>Evaluate the Scikit-Learn and Snap ML Support Vector Machine Models</h2>
</div>


In [ ]:
# compute the Snap ML vs Scikit-Learn training speedup
training_speedup = sklearn_time/snapml_time
print('[Support Vector Machine] Snap ML vs. Scikit-Learn training speedup : {0:.2f}x '.format(training_speedup))

# run inference using the Scikit-Learn model
# get the confidence scores for the test samples
sklearn_pred = sklearn_svm.decision_function(X_test)

# evaluate accuracy on test set
acc_sklearn  = roc_auc_score(y_test, sklearn_pred)
print("[Scikit-Learn] ROC-AUC score:   {0:.3f}".format(acc_sklearn))

# run inference using the Snap ML model
# get the confidence scores for the test samples
snapml_pred = snapml_svm.decision_function(X_test)

# evaluate accuracy on test set
acc_snapml  = roc_auc_score(y_test, snapml_pred)
print("[Snap ML] ROC-AUC score:   {0:.3f}".format(acc_snapml))

As shown above both SVM models provide the same score on the test dataset. However, as in the case of decision trees, Snap ML runs the training routine faster than Scikit-Learn. For more Snap ML examples, please visit [snapml-examples](https://ibm.biz/BdPfxP). Moreover, as shown above, not only is Snap ML seemlessly accelerating scikit-learn applications, but the library's Python API is also compatible with scikit-learn metrics and data preprocessors.

Como se muestra arriba, ambos modelos SVM proporcionan la misma puntuación en el conjunto de datos de prueba. Sin embargo, como en el caso de los árboles de decisión, Snap ML ejecuta la rutina de entrenamiento más rápido que Scikit-Learn. Para obtener más ejemplos de Snap ML, visite [snapml-examples](https://ibm.biz/BdPfxP). Además, como se muestra arriba, Snap ML no solo acelera sin problemas las aplicaciones de scikit-learn, sino que la API de Python de la biblioteca también es compatible con las métricas y los preprocesadores de datos de scikit-learn.


### Practice


In this section you will evaluate the quality of the SVM models trained above using the hinge loss metric (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.hinge_loss.html). Run inference on the test set using both Scikit-Learn and Snap ML models. Compute the hinge loss metric for both sets of predictions. Print the hinge losses of Scikit-Learn and Snap ML.

En esta sección, evaluará la calidad de los modelos SVM entrenados anteriormente utilizando la métrica de pérdida de bisagra (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.hinge_loss.html). Ejecute la inferencia en el conjunto de prueba utilizando los modelos Scikit-Learn y Snap ML. Calcule la métrica de pérdida de bisagra para ambos conjuntos de predicciones. Imprima las pérdidas de bisagra de Scikit-Learn y Snap ML.


In [ ]:
# your code goes here

In [ ]:
# get the confidence scores for the test samples
sklearn_pred = sklearn_svm.decision_function(X_test)
snapml_pred  = snapml_svm.decision_function(X_test)

# import the hinge_loss metric from scikit-learn
from sklearn.metrics import hinge_loss

# evaluate the hinge loss from the predictions
loss_snapml = hinge_loss(y_test, snapml_pred)
print("[Snap ML] Hinge loss:   {0:.3f}".format(loss_snapml))

# evaluate the hinge loss metric from the predictions
loss_sklearn = hinge_loss(y_test, sklearn_pred)
print("[Scikit-Learn] Hinge loss:   {0:.3f}".format(loss_snapml))

# the two models should give the same Hinge loss

## Authors


Andreea Anghel


### Other Contributors


Joseph Santarcangelo


## <h3 align="center">  Copyright &copy; IBM Corporation.  <h3/>
